In [1]:
import json

# logging
import logging
import pathlib

import numpy as np
import mlflow

# Data management
import pandas as pd
from prefect import flow, task

# Models
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

logging.basicConfig(level=logging.INFO)


def read_data() -> pd.DataFrame:
    """Read data into DataFrame"""
    df = pd.read_json("data_raw/data_fifa_players.json")
    df = df.drop(columns=["url"])
    return df


def change_data_types(df):
    """change data types for better manage"""

    # cm
    df["height"] = df["height"].str.extract("(\\d+)").astype(int)
    # kg
    df["weight"] = df["weight"].str.extract("(\\d+)").astype(int)
    # euros
    df["Value"] = pd.to_numeric(df["Value"].str.replace("€|\\.", "", regex=True))
    df["Wage"] = pd.to_numeric(df["Wage"].str.replace("€|\\.", "", regex=True))

    return df


def get_position_zone(df):
    """Transform 'preferred_positions' to 'position_zone'"""
    position_zone = []
    for x in df["preferred_positions"]:
        listb = {"DEFENDING": 0, "MIDFIELD": 0, "ATTACKING": 0, "GOALKEEPER": 0}
        for y in x:
            if y in ["GK"]:
                listb["GOALKEEPER"] = 1
            else:
                if y in ["LF", "RF", "CF", "ST"]:
                    listb["ATTACKING"] = listb["ATTACKING"] + 1
                if y in ["CAM", "RM", "RW", "CDM", "CM", "LM", "LW"]:
                    listb["MIDFIELD"] = listb["MIDFIELD"] + 1
                if y in ["LB", "LWB", "RB", "RWB", "CB"]:
                    listb["DEFENDING"] = listb["DEFENDING"] + 1

        # Keep the position with the highest value
        position_zone.append(max(listb, key=listb.get))

    df.loc[:, "position_zone"] = position_zone
    df = df.drop(columns=["preferred_positions"])

    return df


def one_hot_coding(df):
    """Conver columns type 'object' to one hot coding, then
    delete leaving only the One hot coding"""
    dummies_object = None
    columns_type_object = []

    for i, column_type in enumerate([str(d) for d in df.dtypes]):
        if column_type == "object":
            column_name = df.columns[i]
            columns_type_object.append(column_name)

            dummies = pd.get_dummies(df[column_name], prefix=column_name)

            if dummies_object is None:
                dummies_object = dummies
            else:
                dummies_object = pd.concat([dummies_object, dummies], axis=1)

    df = df.drop(columns_type_object, axis="columns")
    df = pd.concat([df, dummies_object], axis=1)
    return df


def clean_data(df):
    df = df.dropna().copy()
    df = change_data_types(df)
    df = get_position_zone(df)
    df = df.drop(columns=["Wage", "Birth Date", "name", "nation"])
    df = one_hot_coding(df)

    return df


def split_data(df, tsize=0.2):
    y = df["Value"].values  # Target
    # Transform to log
    y = np.log(y)
    y = y.reshape(-1, 1)
    X = df.drop(columns=["Value"])  # Feature(s)
    #scaler = StandardScaler()
    #X_scaler = scaler.fit(X)
    #X = X_scaler.transform(X)
    #y_scaler = scaler.fit(y)
    #y = y_scaler.transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=tsize, random_state=3
    )

    return X_train, X_test, y_train, y_test


def data_pipline(X_train):
    # Para el escalado de las variables numericas usaremos MinMaxScaler
    # y como lo mensionamos anterirormente usaremos el promedio para la imputación
    numeric_pipeline = Pipeline(steps=[
        ('impute', SimpleImputer(strategy='mean')),
        ('StandardScaler', StandardScaler()),
        ('scale', MinMaxScaler()),
    ])
    # handle_unknown='ignore' es importante en caso de tomar una categoria que no se encontraba 
    # durante el proceso de entrenamiento
    categorical_pipeline = Pipeline(steps=[
        ('one-hot', OneHotEncoder(handle_unknown='ignore', sparse=False))
    ])
    # diferenciamos varibles numericas y las que no los son
    numerical_features = X_train.select_dtypes(include=['number']).columns
    categorical_features = X_train.select_dtypes(exclude=['number']).columns
    full_processor = ColumnTransformer(transformers=[
        ('number', numeric_pipeline, numerical_features),
        ('category', categorical_pipeline, categorical_features)
    ])

    return full_processor

def load_parametres() -> dict:
    parameters_path = "parametres.json"
    f = open(parameters_path)
    parameters = json.load(f)
    mlflow.log_artifact(parameters_path, artifact_path="parametres")
    return parameters


def train_best_model(X_train, X_test, y_train, y_test) -> None:
    """train a model with best hyperparams and write everything out"""

    with mlflow.start_run():
        # parameters = {
        #    "param_distributions": {
        #        "max_depth": [3, 6, 10],
        #        "learning_rate": [0.01, 0.05, 0.1, 0.3, 0.5],
        #        "n_estimators": [100, 500, 1000],
        #        "colsample_bytree": [0.7, 0.5],
        #    },
        #    "scoring": "neg_mean_squared_error",
        #    "verbose": 3,
        #    "n_jobs": -1,
        # }
        # Read parametres
        parameters = load_parametres()
        # pipline
        full_processor = data_pipline(X_train)
        ##
        xgb_model = XGBRegressor(seed=20)
        xgb_grid = RandomizedSearchCV(estimator=xgb_model, **parameters)
        xgb_pipeline = Pipeline(steps=[
            ('preprocess', full_processor),
            ('model', xgb_grid)
        ])
        model = xgb_pipeline.fit(X_train, y_train)
        mlflow.log_params(model['model'].best_params_)
        # accuracy = model.score(X_test, y_test)
        predictions = model.predict(X_test)
        logging.debug(f"fit_model.best_params: {model['model'].best_params_}")
        rmse_xgb_reg = mean_squared_error(
            y_test,
            predictions.reshape(-1, 1),
            squared=False,
        )
        print(rmse_xgb_reg)
        logging(f"The RMSE for xgb_reg is: {rmse_xgb_reg}")
        logging(f"Best params are: {model['model'].best_params_}")

        mlflow.log_metric("rmse", rmse_xgb_reg)
        #pathlib.Path("models").mkdir(exist_ok=True)
        mlflow.sklearn.log_model(model, artifact_path="models_mlflow")
        print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")
    return None


In [2]:
# def main_flow() -> None:
#    """The main training pipeline"""
# MLflow settings
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("xgb_reg")
# Load
df = read_data()
# Transform
df = clean_data(df)
X_train, X_test, y_train, y_test = split_data(
    df,
    tsize=0.2,
)
# Train
train_best_model(X_train, X_test, y_train, y_test)



d:\Alonmar\Documents\pruebas\dvc_mflow_prefect\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fitting 5 folds for each of 10 candidates, totalling 50 fits


TypeError: 'module' object is not callable

In [196]:
mlflow.search_experiments()[0]

<Experiment: artifact_location='file:///D:/Alonmar/Documents/pruebas/dvc_mflow_prefect/mlruns/1', creation_time=1696493101498, experiment_id='1', last_update_time=1696493101498, lifecycle_stage='active', name='xgb_reg', tags={}>

In [165]:
def load_model(run_id):
   logged_model = f'runs:/{run_id}/models_mlflow'
   model = mlflow.pyfunc.load_model(logged_model)
   return model

In [143]:
def scale_input(pX, run_id):
    """Scales values for input"""
    #input_scaler = StandardScaler().fit(df)
    logged_model = f'runs:/{run_id}/models_mlflow/'
    input_scaler = mlflow.artifacts.download_artifacts()
    scaled_input = input_scaler.transform(pX)
    return scaled_input

In [144]:
def scale_target(target, df):
    """Scales Target 'y' Value"""
    y = df["Value"].values  # Target
    y = np.log(y)
    y = y.reshape(-1, 1)  # Reshape
    # model['preprocess'].transformers[0][1]['StandardScaler']
    scaler = StandardScaler()
    y_scaler = scaler.fit(y)
    scaled_target = y_scaler.inverse_transform(target)
    return scaled_target

In [145]:
def human_readable_payload(value_predict):
    """Takes numpy array and returns back human readable dictionary"""

    value_log = float(np.round(value_predict, 2))
    value_stimate = float(np.round(np.exp(value_predict), 2))
    result = {
        "value_log": value_log,
        "value_money": f"{value_stimate} euros",
    }
    return result

In [181]:
def predict(pX: dict, run_id: str) -> dict:
    """Takes weight and predicts height"""

    model = load_model(run_id)
    df = pd.DataFrame(pX)
    #df = clean_data(pd.DataFrame(test_data))
    prediction = model.predict(df)

    result = human_readable_payload(prediction)
    return result

In [147]:
test_data = [{"overallrating": 75,
  "potencial": 76,
  "height": 174,
  "weight": 67,
  "Age": 26,
  "Contract Length": 2024,
  "Ball Control": 78.0,
  "Dribbling": 81.0,
  "Marking": 32.0,
  "Slide Tackle": 26.0,
  "Stand Tackle": 30.0,
  "Aggression": 61.0,
  "Reactions": 73.0,
  "Att. Position": 72.0,
  "Interceptions": 49.0,
  "Vision": 70.0,
  "Composure": 79.0,
  "Crossing": 66.0,
  "Short Pass": 71.0,
  "Long Pass": 63.0,
  "Acceleration": 83.0,
  "Stamina": 76.0,
  "Strength": 55.0,
  "Balance": 86.0,
  "Sprint Speed": 75.0,
  "Agility": 89.0,
  "Jumping": 76.0,
  "Heading": 50.0,
  "Shot Power": 71.0,
  "Finishing": 76.0,
  "Long Shots": 73.0,
  "Curve": 73.0,
  "FK Acc.": 69.0,
  "Penalties": 72.0,
  "Volleys": 70.0,
  "GK Positioning": 15,
  "GK Diving": 9,
  "GK Handling": 12,
  "GK Kicking": 8,
  "GK Reflexes": 14,
  "Preferred Foot_Left": 0,
  "Preferred Foot_Right": 1,
  "Player Work Rate_High / High": 0,
  "Player Work Rate_High / Low": 0,
  "Player Work Rate_High / Medium": 1,
  "Player Work Rate_Low / High": 0,
  "Player Work Rate_Low / Low": 0,
  "Player Work Rate_Low / Medium": 0,
  "Player Work Rate_Medium / High": 0,
  "Player Work Rate_Medium / Low": 0,
  "Player Work Rate_Medium / Medium": 0,
  "position_zone_ATTACKING": 0,
  "position_zone_DEFENDING": 0,
  "position_zone_GOALKEEPER": 0,
  "position_zone_MIDFIELD": 1}]

In [185]:
run_id = 'fe5bf6d4c7c14a428ed7dc9cb4d1cb09'
#logged_model = f'runs:/{run_id}/models_mlflow'
#load_model = mlflow.pyfunc.load_model(logged_model)

In [216]:
TRACKING_URL = "http://127.0.0.1:5000"

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

mlflow.set_tracking_uri(TRACKING_URL)

In [186]:
predict(pX=test_data, run_id=run_id)

C:\Users\vocho\AppData\Local\Temp\ipykernel_4568\1462953935.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  value_log = float(np.round(value_predict, 2))
C:\Users\vocho\AppData\Local\Temp\ipykernel_4568\1462953935.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  value_stimate = float(np.round(np.exp(value_predict), 2))


{'value_log': 15.619999885559082, 'value_money': '6050965.0 euros'}